# IMPORT API KEY

In [1]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

# TAVILY SEARCH
- [https://docs.langchain.com/oss/python/integrations/providers/tavily]
- [https://docs.langchain.com/oss/python/integrations/tools/tavily_search]
- [https://docs.langchain.com/oss/python/integrations/tools/tavily_extract]

In [4]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [5]:
answer = tool.invoke({"query": "What happened at the last wimbledon"})

In [6]:
answer

{'query': 'What happened at the last wimbledon',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.cnn.com/sport/live-news/wimbledon-final-mens-25-07-13-spt',
   'title': 'Wimbledon final highlights: Jannik Sinner beats Carlos Alcaraz - CNN',
   'content': '# Jannik Sinner beats Carlos Alcaraz to win his first Wimbledon title. • **Men’s singles champion:** World No. 1 Jannik Sinner won his first Wimbledon title in a hard-fought battle with two-time reigning champion Carlos Alcaraz. Jannik Sinner is the men’s singles champion at Wimbledon for the first time after he came from a set down to beat two-time defending champion Carlos Alcaraz, 4-6, 6-4, 6-4, 6-4. Sinner looked to have taken control in the first set when he broke his opponent’s serve in the fifth game to take a 3-2 lead, only for Alcaraz to break back twice — the second time thanks in part to a suddenly nervous looking Sinner, who double-faulted to give the Spaniard an advantage, w

In [11]:
for result in answer.get("results"):
    print(f"{result.get('score')}-----{result.get('title')}")
    print(result.get('url'))
    print(f"{result.get('content')[:200]}...")
    print("\n\n")

0.9695556-----Wimbledon final highlights: Jannik Sinner beats Carlos Alcaraz - CNN
https://www.cnn.com/sport/live-news/wimbledon-final-mens-25-07-13-spt
# Jannik Sinner beats Carlos Alcaraz to win his first Wimbledon title. • **Men’s singles champion:** World No. 1 Jannik Sinner won his first Wimbledon title in a hard-fought battle with two-time reign...



0.51952136-----Jannik Sinner defeats Carlos Alcaraz for 1st Wimbledon title
https://sports.yahoo.com/tennis/live/2025-wimbledon-mens-singles-final-jannik-sinner-defeats-carlos-alcaraz-for-1st-wimbledon-title-140016594.html
In the fourth set, Sinner took an early 3-1 advantage by breaking Alcaraz's serve, winning two consecutive backhands that sliced down the line and out of reach. While Alcaraz held to fight back in the...



0.4016878-----What were the Wimbledon results? | ATP Tour | Tennis
https://www.atptour.com/en/news/wimbledon-2025-results
# ATP Tour. #### TOURNAMENT RESULTS. #### PLAYER RESULTS. ## What were the Wimbledon res

# TAVILY SEARCH PARAMETERS
- [https://docs.tavily.com/documentation/api-reference/endpoint/search]
- [https://docs.tavily.com/documentation/best-practices/best-practices-search]

## auto_parameters

Tavily automatically configures parameters based on query intent. Your explicit values override automatic ones.
```
{
  "query": "impact of AI in education policy",
  "auto_parameters": true,
  "search_depth": "basic" // Override to control cost
}
```
auto_parameters may set search_depth to advanced (2 credits). Set it manually to control cost.

## search_depth

- **ultra-fast**: When latency is absolutely crucial. Delivers near-instant results, prioritizing speed over relevance. Ideal for real-time applications where response time is critical.
- **fast**: When latency is more important than relevance, but you want results in reranked chunks format. Good for applications that need quick, targeted snippets.
- **basic**: A solid balance between relevance and latency. Best for general-purpose searches where you need quality results without the overhead of advanced processing.
- **advanced**:	When you need the highest relevance and are willing to trade off latency. Best for queries seeking specific, detailed information.

```
{
  "query": "How many countries use Monday.com?",
  "search_depth": "advanced",
  "chunks_per_source": 3,
  "include_raw_content": true
}
```

| Depth     | Latency | Relevance | Content Type |
|-----------|---------|-----------|--------------|
| ultra-fast| Lowest  | Lower     | Content      |
| fast      | Low     | Good      | Chunks       |
| basic     | Medium  | High      | Content      |
| advanced  | Higher  | Highest   | Chunks       |

| Type    | Description                                             |
|---------|---------------------------------------------------------|
| Content | NLP-based summary of the page, providing general context |
| Chunks  | Short snippets reranked by relevance to your search query |

- Use **chunks** when you need highly targeted information aligned with your query. 
- Use **content** when a general page summary is sufficient.

## exact_match

Use exact_match only when searching for a specific name or phrase that must appear verbatim in the source content. Wrap the phrase in quotes within your query:

```
{
  "query": "\"John Smith\" CEO Acme Corp",
  "exact_match": true
}
```

Because this narrows retrieval, it may return fewer results or empty result fields when no exact matches are found. Best suited for:
- **Due diligence** — finding information on a specific person or entity
- **Data enrichment** — retrieving details about a known company or individual
- **Legal/compliance** — locating exact names or phrases in public records

## Filtering Results: By date

| Parameter          | Description                                         |
|-------------------|-----------------------------------------------------|
| time_range         | Filter by relative time: day, week, month, year    |
| start_date / end_date | Filter by specific date range (format: YYYY-MM-DD) |

## Filtering Results: By domain

| Parameter       | Description                         |
|-----------------|-------------------------------------|
| include_domains | Limit to specific domains            |
| exclude_domains | Filter out specific domains          |
| country         | Boost results from a specific country |

```
// Restrict to LinkedIn profiles
{ "query": "CEO background at Google", "include_domains": ["linkedin.com/in"] }

// Exclude irrelevant domains
{ "query": "US economy trends", "exclude_domains": ["espn.com", "vogue.com"] }

// Boost results from a country
{ "query": "tech startup funding", "country": "united states" }

// Wildcard: limit to .com, exclude specific site
{ "query": "AI news", "include_domains": ["*.com"], "exclude_domains": ["example.com"] }
```

In [14]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=False,
    include_raw_content=False,
    include_images=False,
    include_image_descriptions=False,
    search_depth="basic",
    time_range="year",
    include_domains=["*.it"], #default: None
    exclude_domains=["*.com", "wikipedia.org"], #default: None
    country="italy"
)

answer = tool.invoke({"query": "\"Pasta alla carbonara\""})

In [15]:
answer

{'query': 'Pasta alla carbonara',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.informacibo.it/carbonara-cacio-e-pepe-gricia-e-amatriciana-ricette-con-pecorino-romano/',
   'title': 'Carbonara, cacio e pepe, gricia e amatriciana: ricette con pecorino ...',
   'content': 'La pasta alla carbonara è probabilmente la ricetta romana più famosa al mondo. Qui il pecorino romano lavora insieme ai tuorli per creare una crema stabile e',
   'score': 0.5114976,
   'raw_content': None},
  {'url': 'https://blog.giallozafferano.it/ilcaldosaporedelsud/pasta-alla-carbonara/',
   'title': 'Pasta alla carbonara: ricetta originale romana cremosa',
   'content': 'Pasta alla carbonara: la ricetta tradizionale romana, cremosa e senza panna con guanciale e pecorino. Scopri come prepararla passo passo.',
   'score': 0.4572585,
   'raw_content': None},
  {'url': 'https://www.italyshopservice.it/blog/gustories-1/carbonara-2',
   'title': 'Carbonara - Italy Shop 

In [16]:
for result in answer.get("results"):
    print(f"{result.get('score')}-----{result.get('title')}")
    print(result.get('url'))
    print(f"{result.get('content')[:200]}...")
    print("\n\n")

0.5114976-----Carbonara, cacio e pepe, gricia e amatriciana: ricette con pecorino ...
https://www.informacibo.it/carbonara-cacio-e-pepe-gricia-e-amatriciana-ricette-con-pecorino-romano/
La pasta alla carbonara è probabilmente la ricetta romana più famosa al mondo. Qui il pecorino romano lavora insieme ai tuorli per creare una crema stabile e...



0.4572585-----Pasta alla carbonara: ricetta originale romana cremosa
https://blog.giallozafferano.it/ilcaldosaporedelsud/pasta-alla-carbonara/
Pasta alla carbonara: la ricetta tradizionale romana, cremosa e senza panna con guanciale e pecorino. Scopri come prepararla passo passo....



0.4429021-----Carbonara - Italy Shop Service
https://www.italyshopservice.it/blog/gustories-1/carbonara-2
Carbonara is a dish that divides, excites, and unites. It is the symbol of Roman cuisine, but also a small gastronomic battlefield where every detail counts....



0.4292146-----Come fare la carbonara perfetta - Sonia Peronaci
https://www.soniaperonaci.it/p

# SEARCH AGENT

In [2]:
from typing import List, Literal, Optional
from langchain.tools import tool
from langchain_tavily import TavilySearch

@tool
def tavily_search_no_structured(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    include_image_descriptions: bool = False,
    search_depth: Literal["basic", "advanced"] = "basic",
    # time_range: Optional[Literal["day", "week", "month", "year"]] = None,
    # include_domains: Optional[List[str]] = None,
    # exclude_domains: Optional[List[str]] = None,
    # country: Optional[str] = None
    time_range: str = "",            # instead of None
    include_domains: List[str] = [], # instead of None
    exclude_domains: List[str] = [], # instead of None
    country: str = ""                # instead of None
) -> str:
    """
    Perform a web search using TavilySearch.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Search results as a string.
    """
    # parameters validation
    if len(time_range) == 0:
        time_range = None
    if len(include_domains) == 0:
        include_domains = None
    if len(exclude_domains) == 0:
        exclude_domains = None
    if len(country) == 0:
        country = None

    tool_instance = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        include_image_descriptions=include_image_descriptions,
        search_depth=search_depth,
        time_range=time_range,
        include_domains=include_domains,
        exclude_domains=exclude_domains,
        country=country
    )

    return tool_instance.invoke({"query": query})

In [3]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

search_agent = create_agent(
    model=model,
    tools=[tavily_search_no_structured],
    system_prompt = """
        You are a helpful assistant. You can execute the 'tavily_search' tool to perform a web search. 
        
        - The user will always provide a dictionary of parameters for the tool.
        - Only use the parameters provided by the user.
        - Execute the tool using these parameters.
        - The user's input will be in two parts: 'text' (query) and 'params' (dictionary of tool parameters).
        """,
)

In [4]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_params}
params_json = json.dumps(params)

user_query = "\"Pasta alla carbonara\""
query = f"text: {user_query}, params: {params_json}"

result = search_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
)

In [5]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='077e5387-c058-438e-bb8e-22527aa9430b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-03T16:50:20.9595315Z', 'done': True, 'done_reason': 'stop', 'total_duration': 342048817000, 'load_duration': 10712957700, 'prompt_eval_count': 697, 'prompt_eval_duration': 56739927600, 'eval_count': 161, 'eval_duration': 273616293500, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d543b-8070-7c61-9cce-ed6696ac1559-0', tool_calls=[{'name': 'tavily_search_no_structured', 'args': {'topic': 'general', 'inc

# SEARCH AGENT - STRUCTURED OUTPUT

In [6]:
from typing import List, Literal, Optional
from pydantic import BaseModel, HttpUrl, Field
from langchain.tools import tool
from langchain_tavily import TavilySearch

class TavilySearchResult(BaseModel):
    url: HttpUrl = Field(description="The URL of the search result")
    title: str = Field(description="The title of the result page")
    content: str = Field(description="The main content snippet of the result")
    raw_content: Optional[str] = Field(default=None, description="Raw HTML/content if available")
    score: float = Field(description="Relevance score of the result")

class TavilySearchOutput(BaseModel):
    query: str = Field(description="The search query string")
    follow_up_questions: Optional[List[str]] = Field(default=None, description="Suggested follow-up questions")
    answer: Optional[str] = Field(default=None, description="Direct answer if requested")
    images: List[str] = Field(default_factory=list, description="List of images related to the query")
    results: List[TavilySearchResult] = Field(default_factory=list, description="List of structured search results")
    response_time: float = Field(description="Time taken to complete the search in seconds")
    request_id: str = Field(description="Unique request identifier")

@tool
def tavily_search(
    query: str,
    max_results: int = 5,
    topic: str = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    search_depth: str = "basic",
    # time_range: Optional[str] = None,
    # include_domains: Optional[List[str]] = None,
    # exclude_domains: Optional[List[str]] = None,
    # country: Optional[str] = None
    time_range: str = "",            # instead of None
    include_domains: List[str] = [], # instead of None
    exclude_domains: List[str] = [], # instead of None
    country: str = ""                # instead of None
) -> TavilySearchOutput:
    """
    Perform a web search using TavilySearch and return structured results.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Structured output as TavilySearchOutput
    """
    # parameters validation
    if len(time_range) == 0:
        time_range = None
    if len(include_domains) == 0:
        include_domains = None
    if len(exclude_domains) == 0:
        exclude_domains = None
    if len(country) == 0:
        country = None

    search_tool = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        search_depth=search_depth,
        time_range=time_range,
        include_domains=include_domains,
        exclude_domains=exclude_domains,
        country=country
    )

    raw_output = search_tool.invoke({"query": query})

    structured_results = [
        TavilySearchResult(
            url=item["url"],
            title=item["title"],
            content=item["content"],
            raw_content=item.get("raw_content") if include_raw_content else None,
            score=item.get("score", 0.0)
        )
        for item in raw_output.get("results", [])
    ]

    return TavilySearchOutput(
        query=raw_output.get("query", query),
        follow_up_questions=raw_output.get("follow_up_questions"),
        answer=raw_output.get("answer") if include_answer else None,
        images=raw_output.get("images", []) if include_images else [],
        results=structured_results,
        response_time=raw_output.get("response_time", 0.0),
        request_id=raw_output.get("request_id", "")
    )

In [7]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

search_agent = create_agent(
    model=model,
    tools=[tavily_search],
    response_format = ToolStrategy(TavilySearchOutput),
    system_prompt = """
        You are a research assistant. You can execute the 'tavily_search' tool to perform a web search.
        1. Use the 'tavily_search' tool to get results.
        2. The user will always provide a dictionary of parameters for the tool. Only use the parameters provided by the user. 
        Execute the tool using these parameters.
        3. The user's input will be in two parts: 'text' (query) and 'params' (dictionary of tool parameters).
        4. Your final response MUST be the EXACT object you received from the tool.
        5. You MUST transfer every single item from the 'results' list of the tool output into the 'results' field 
        of your final structured response. DO NOT omit any information.
        """,
)

In [8]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_params}
params_json = json.dumps(params)

user_query = "\"Pasta alla carbonara\""
query = f"text: {user_query}, params: {params_json}"

result = search_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
)

In [9]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='68393000-e4c6-4744-a8d4-8e3def3a8433'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-03T17:15:11.8473392Z', 'done': True, 'done_reason': 'stop', 'total_duration': 185228732100, 'load_duration': 11080442400, 'prompt_eval_count': 1094, 'prompt_eval_duration': 78695754100, 'eval_count': 228, 'eval_duration': 95004722500, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d5454-a4d8-71d2-833e-8d3d4d243840-0', tool_calls=[{'name': 'tavily_search', 'args': {'max_results': 5, 'topic': 'general', 

In [11]:
type(result["messages"][-1].content)

str